# Industry Intelligence Pipeline: fractional CFO services

**What this is.** A pipeline that takes a company description, picks a NAICS code from the official Census manual,
ingests industry-report PDFs (IBISWorld, First Research) that were downloaded by hand, extracts seven structured
"signals" with a page-level citation on every one, **verifies every quote against the PDF page in code**, reconciles
the documents, lists the gaps, and writes a brief.

**Who did what (read this first).**
* The *extraction* step (reading pages, writing `extractions/*.json`) was performed by **Claude Code, an LLM agent**, during the build
  session. No paid API was called, and no API key was used.
* Everything else in this notebook is ordinary Python that runs when you execute it: PDF text extraction, the NAICS fetch,
  quote verification, cross-document reconciliation, gap analysis and brief generation.
* The verifier checks that a quote **exists on the cited page**. It does not check that the LLM interpreted the quote correctly.

**Licensing.** The reports are paywalled. They are not in the repository. Only short quotes (<= 25 words) appear in committed files.
Without the PDFs, Stages 0 and 3 cannot be re-executed. The notebook then replays the saved verification log and says so.

**Design rule used throughout:** each stage has a markdown cell above it stating the choice, the rejected alternative, and why.

In [1]:
# ---- CONFIG: edit this cell to run a different industry -------------------------------------------------------------
import os, sys, re, json, glob, time, hashlib, random, datetime, logging, warnings, textwrap, collections, unicodedata
from pathlib import Path

if os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError("ANTHROPIC_API_KEY is set. This project is zero-cost by design: unset it and re-run.")

logging.getLogger("pdfminer").setLevel(logging.ERROR)   # pdfplumber is chatty on unusual PDF colour spaces
warnings.filterwarnings("ignore")

COMPANY      = "Working name TBD, fractional CFO services"
DESCRIPTION  = ("Fractional (part-time, outsourced) CFO services for US small and mid-sized businesses with roughly "
                "$2M-$20M revenue. The company does not exist yet (founders are MAcc students), so the brief describes "
                "the industry we would enter.")
REPORTS_DIR  = "reports"            # paywalled PDFs, never committed
CACHE_DIR    = "cache"              # page text cache, never committed
EXTRACT_DIR  = "extractions"        # one JSON per document (committed; quotes are <= 25 words)
EXTRACTOR    = "claude_code_files"  # or "gemini" (implemented, untested; see Stage 2)
GEMINI_MODEL = None                 # None = pick from the live model list at run time (see Stage 2 backend)

# Candidate NAICS codes for the choice in Stage 1 (Census 2022 manual).
NAICS_CANDIDATES = ["541611", "541219", "541211", "541618", "541990", "561110"]
NAICS_MANUAL_URL = "https://www.census.gov/naics/reference_files_tools/2022_NAICS_Manual.pdf"
NAICS_LANDING    = "https://www.census.gov/naics/"

# The NAICS choice is a judgement, so it lives here where you can change it. Stage 1 re-validates it: every phrase in JUSTIFICATION_QUOTES
# must appear in the fetched manual entry of that code, or the notebook stops.
CHOSEN_CODE, NEIGHBOR_CODE = "541611", "541219"
JUSTIFICATION_QUOTES = {
    "541611": ["financial planning and budgeting", "Financial management (except investment advice) consulting services",
               "providing operating advice and assistance to businesses"],
    "541219": ["providing accounting services (except tax return preparation services only or payroll services only)",
               "Accountant (except CPA) offices, bookkeeper offices, and billing offices are included"],
}
NAICS_JUSTIFICATION = (
    "We chose 541611 because the manual defines it as \"providing operating advice and assistance to businesses\" on issues such as "
    "\"financial planning and budgeting\" and lists \"Financial management (except investment advice) consulting services\" as an example, "
    "which describes advice to a client's management; the closest neighbor, 541219, is instead defined as \"providing accounting services "
    "(except tax return preparation services only or payroll services only)\" for establishments outside CPA offices, and its own text says "
    "\"Accountant (except CPA) offices, bookkeeper offices, and billing offices are included\", so it is organized around producing the books rather than advising management on them."
)
print("Company:", COMPANY)
print("Reports dir:", REPORTS_DIR, "| PDFs found:", len(glob.glob(f"{REPORTS_DIR}/*.pdf")))

Company: Working name TBD, fractional CFO services
Reports dir: reports | PDFs found: 3


## Stage 0: Ingest (PDF to page-level text cache)

**Choice:** extract text *per PDF page* with `pdfplumber`, and cache `doc_id`, `pdf_page`, printed page label and text in
`cache/pages.jsonl`. Two text variants are cached per page: `text` (column-aware: two-column IBISWorld pages are
re-ordered so a sentence is contiguous) and `text_raw` (pdfplumber's default reading order).

**Alternative rejected:** `pypdf` alone, or one text blob per document. `pypdf` merges columns and drops spacing more often;
one blob loses the page number, and a citation without a page is not findable. Whole-document OCR was also rejected
(nothing here is scanned; the code flags near-empty pages and would try OCR if `pytesseract` were installed).

**Why two variants:** IBISWorld reports are two-column. Default extraction interleaves the columns line by line, which breaks
verbatim quotes that wrap across lines. The verifier accepts a quote if it appears in *either* variant of the cited page.
**Limit:** tables and charts extract poorly in both variants; numbers taken from them are checked against the page text only.

In [2]:
import pdfplumber

PRINTED_LABEL_PATTERNS = [
    re.compile(r"^(\d{1,3})\s+www\.ibisworld\.com\b"),   # IBISWorld footer: "17 www.ibisworld.com May 2026"
    re.compile(r"(\d{1,3})/\d{1,3}\s*$"),                # browser-printout footer: "10/26"
]

def _cluster_lines(words, tol=3):
    lines, cur, cur_top = [], [], None
    for w in sorted(words, key=lambda w: (round(w["top"]), w["x0"])):
        if cur_top is None or abs(w["top"] - cur_top) <= tol:
            cur.append(w); cur_top = w["top"] if cur_top is None else cur_top
        else:
            lines.append(cur); cur, cur_top = [w], w["top"]
    if cur: lines.append(cur)
    return [sorted(l, key=lambda w: w["x0"]) for l in lines]

def column_aware_text(page, band=0.05):
    """Re-order a page so two-column blocks read left column then right column. Header/footer bands are dropped."""
    H, mid = float(page.height), float(page.width) / 2
    words = [w for w in page.extract_words() if H * band <= w["top"] <= H * (1 - band)]
    out, L, R = [], [], []
    def flush():
        out.extend(L); out.extend(R); L.clear(); R.clear()
    for line in _cluster_lines(words):
        crossing = any(w["x0"] < mid - 2 and w["x1"] > mid + 2 for w in line)
        left  = [w for w in line if (w["x0"] + w["x1"]) / 2 <  mid]
        right = [w for w in line if (w["x0"] + w["x1"]) / 2 >= mid]
        if crossing or (left and right and right[0]["x0"] - left[-1]["x1"] < 8):
            flush(); out.append(" ".join(w["text"] for w in line))          # full-width line
        else:
            if left:  L.append(" ".join(w["text"] for w in left))
            if right: R.append(" ".join(w["text"] for w in right))
    flush()
    return "\n".join(out)

def printed_label(raw_text):
    lines = [l.strip() for l in raw_text.strip().split("\n") if l.strip()]
    for l in lines[-3:][::-1]:
        for pat in PRINTED_LABEL_PATTERNS:
            m = pat.search(l)
            if m: return m.group(1)
    return None

def detect_doc_meta(pages, filename):
    """Title, publisher, publication date and stated industry code, from the front pages (and, for First Research, the code page)."""
    front = "\n".join(p["text_raw"] for p in pages[:2])
    whole = "\n".join(p["text_raw"] for p in pages)
    meta = {"filename": filename, "n_pages": len(pages)}
    if "ibisworld" in front.lower():
        meta["publisher"] = "IBISWorld"
        m = re.search(r"Services\s*•\s*([0-9]{5}[a-z]?)", front)
        meta["stated_industry_code"] = m.group(1) if m else None
        lines = [l.strip() for l in pages[0]["text_raw"].split("\n") if l.strip()]
        meta["title"] = " ".join(lines[1:3]).replace("the US Accounts", "the US") if len(lines) > 2 else filename
        m = re.search(r"Published:\s*([A-Za-z]+ \d{4})", front)
        meta["publication_date"] = m.group(1) if m else None
        meta["sector_header"] = lines[0]
        meta["how_to_find"] = "BYU Library > Business databases > IBISWorld > search the report title (US industry reports)."
    elif "first research" in front.lower() or "hoovers.dnb.com" in whole.lower():
        meta["publisher"] = "First Research (Dun & Bradstreet), accessed via D&B Hoovers"
        t = pages[0]["text_raw"]
        m = re.search(r"Powered by\s*\n?\s*([A-Z][A-Za-z &,\-]+)", t)
        meta["title"] = "First Research Industry Profile: " + (m.group(1).strip() if m else "Accounting Services")
        m = re.search(r"Published:\s*([A-Za-z]+ \d{4})", front)
        meta["publication_date"] = m.group(1) if m else None
        code_page = next((p for p in pages if "Associated Industry Codes" in p["text_raw"]), None)
        m = re.search(r"NAICS[^\n]{0,20}?(\d{5,6})", code_page["text_raw"]) if code_page else None
        meta["stated_industry_code"] = m.group(1) if m else None
        meta["code_note"] = ("The 'Associated Industry Codes' section (PDF p.%s) is empty in the printout; no NAICS/SIC code is stated. "
                             "Scope is inferred from the title only." % (code_page["pdf_page"] if code_page else "?")) if not m else None
        meta["how_to_find"] = "BYU Library > Business databases > D&B Hoovers > Industries > First Research > 'Accounting Services' (Industry Overview)."
    else:
        meta.update(publisher="unknown", title=filename, publication_date=None, stated_industry_code=None, how_to_find="unknown")
    return meta

def slug_doc_id(meta):
    pub = "firstresearch" if "first research" in meta["publisher"].lower() else re.sub(r"[^a-z]", "", meta["publisher"].lower().split()[0])
    code = (meta.get("stated_industry_code") or "").lower()
    if not code:   # no code stated: fall back to the title after the colon
        code = re.sub(r"[^a-z0-9]+", "-", meta["title"].split(":")[-1].lower()).strip("-")
    return f"{pub}_{code}"

def stage0_ingest(reports_dir=REPORTS_DIR, cache_dir=CACHE_DIR):
    """Extract per-page text for every PDF. Writes cache/pages.jsonl and returns (pages, docs). Flags near-empty pages."""
    os.makedirs(cache_dir, exist_ok=True)
    pages, docs, flags = [], {}, []
    for f in sorted(glob.glob(os.path.join(reports_dir, "*.pdf"))):
        with pdfplumber.open(f) as pdf:
            doc_pages = []
            for i, pg in enumerate(pdf.pages, start=1):
                raw = pg.extract_text() or ""
                doc_pages.append({"pdf_page": i, "printed_page": printed_label(raw), "text": column_aware_text(pg), "text_raw": raw})
        meta = detect_doc_meta(doc_pages, os.path.basename(f))
        doc_id = slug_doc_id(meta)
        meta["doc_id"] = doc_id
        meta["sha256"] = hashlib.sha256(Path(f).read_bytes()).hexdigest()[:16]
        near_empty = [p["pdf_page"] for p in doc_pages if len(p["text_raw"].strip()) < 50]
        meta["near_empty_pages"] = near_empty
        if near_empty:
            flags.append((doc_id, near_empty))
            try:
                import pytesseract  # noqa: F401
                meta["ocr"] = "pytesseract available but OCR path not exercised in this run"
            except ImportError:
                meta["ocr"] = "no OCR engine installed; these pages have no text"
        for p in doc_pages: p["doc_id"] = doc_id
        docs[doc_id] = meta
        pages.extend(doc_pages)
    with open(os.path.join(cache_dir, "pages.jsonl"), "w") as fh:
        for p in pages:
            fh.write(json.dumps({k: p[k] for k in ("doc_id", "pdf_page", "printed_page", "text", "text_raw")}) + "\n")
    # Metadata only (no report text) is safe to commit and feeds the source list.
    with open("sources.json", "w") as fh:
        json.dump(docs, fh, indent=2)
    return pages, docs, flags

def load_pages_cache(cache_dir=CACHE_DIR):
    with open(os.path.join(cache_dir, "pages.jsonl")) as fh:
        return [json.loads(l) for l in fh]

In [3]:
if glob.glob(f"{REPORTS_DIR}/*.pdf"):
    PAGES, DOCS, FLAGS = stage0_ingest()
    print(f"Ingested {len(DOCS)} PDFs, {len(PAGES)} pages -> {CACHE_DIR}/pages.jsonl (gitignored)")
    for d, m in DOCS.items():
        print(f"\n[{d}] {m['title']}\n   publisher: {m['publisher']}\n   published: {m['publication_date']} | stated industry code: {m['stated_industry_code']} | pages: {m['n_pages']}"
              f" | printed-page labels found on {sum(1 for p in PAGES if p['doc_id']==d and p['printed_page'])} pages")
        if m.get("code_note"): print("   NOTE:", m["code_note"])
    print("\nNear-empty (possibly scanned) pages:", FLAGS or "none")
else:
    PAGES, DOCS, FLAGS = [], json.load(open("sources.json")) if os.path.exists("sources.json") else {}, []
    print("No PDFs in", REPORTS_DIR, "- Stage 0 skipped; using committed sources.json (metadata only).")

Ingested 3 PDFs, 138 pages -> cache/pages.jsonl (gitignored)

[ibisworld_54121c] Accounting Services in the US
   publisher: IBISWorld
   published: May 2026 | stated industry code: 54121c | pages: 56 | printed-page labels found on 52 pages

[ibisworld_54161] Management Consulting in the US
   publisher: IBISWorld
   published: August 2026 | stated industry code: 54161 | pages: 56 | printed-page labels found on 52 pages

[firstresearch_accounting-services] First Research Industry Profile: Accounting Services
   publisher: First Research (Dun & Bradstreet), accessed via D&B Hoovers
   published: July 2025 | stated industry code: None | pages: 26 | printed-page labels found on 26 pages
   NOTE: The 'Associated Industry Codes' section (PDF p.26) is empty in the printout; no NAICS/SIC code is stated. Scope is inferred from the title only.

Near-empty (possibly scanned) pages: none


## Stage 1: NAICS selection from the official 2022 NAICS Manual

**Problem:** fractional CFO is not its own NAICS industry, so every number in every report describes a *proxy* industry.
The choice of proxy should come from the classification system's own definitions, not from what sounds right.

**Choice:** fetch the official *2022 NAICS Manual* PDF from census.gov, parse the entry for each candidate code (title,
description, illustrative examples, cross-references), save the text with URL and access date to `naics/`, and pick one code
by comparing the manual's own language. The justification sentence below quotes the manual, and the code **asserts that each quoted phrase
appears in the fetched entry**; a mis-quote fails the notebook.

**Alternatives rejected:** (1) the census.gov/naics search page (returns 403 to scripted requests, so it cannot be a reproducible
source), (2) choosing the code from the report titles (that would let the available data pick the industry), (3) inventing definitions from memory.
If the fetch fails, the code stops and prints what to paste instead of fabricating text.

In [4]:
import requests
from pypdf import PdfReader

NAICS_DIR = "naics"
_FURNITURE = re.compile(r"^(PROFESSIONAL, SCIENTIFIC, AND TECHNICAL SERVICES \d+|\d+ NORTH AMERICAN INDUSTRY CLASSIFICATION SYSTEM|"
                        r"T.Canadian, Mexican, and United States industries are comparable\.|census\.gov/naics|\d{3})\s*$")

def fetch_naics_manual(cache_dir=CACHE_DIR):
    path = os.path.join(cache_dir, "2022_NAICS_Manual.pdf")
    if not os.path.exists(path) or os.path.getsize(path) < 1_000_000:
        os.makedirs(cache_dir, exist_ok=True)
        r = requests.get(NAICS_MANUAL_URL, headers={"User-Agent": "Mozilla/5.0 (class project; academic use)"}, timeout=120)
        if r.status_code != 200 or not r.content.startswith(b"%PDF"):
            raise RuntimeError(f"Could not fetch the NAICS manual ({r.status_code}). Open {NAICS_LANDING}, search each candidate code, "
                               f"and paste the title, description and cross-references into naics/manual_paste.txt instead of relying on memory.")
        Path(path).write_bytes(r.content)
    accessed = datetime.date.fromtimestamp(os.path.getmtime(path)).isoformat()
    return path, accessed

def manual_entries(path, codes, page_range=(430, 520)):
    """Return {code: entry_text} by locating the 6-digit heading and reading to the next code heading."""
    reader = PdfReader(path)
    def scan(lo, hi):
        lines = []
        for i in range(lo - 1, min(hi, len(reader.pages))):
            for l in (reader.pages[i].extract_text() or "").split("\n"):
                l = l.rstrip()
                if l.strip() and not _FURNITURE.match(l.strip()): lines.append((i + 1, l))
        return lines
    for rng in (page_range, (1, len(reader.pages))):          # narrow scan first, whole manual as fallback
        lines = scan(*rng)
        out, head = {}, re.compile(r"^(\d{2,6})\s+([A-Z][^\n]*?)T?\s*$")
        idx = [(n, i, head.match(l)) for i, (n, l) in enumerate(lines)]
        idx = [(n, i, m) for n, i, m in idx if m]
        for c in codes:
            for k, (n, i, m) in enumerate(idx):
                if m.group(1) == c:
                    end = idx[k + 1][1] if k + 1 < len(idx) else len(lines)
                    body = "\n".join(l for _, l in lines[i + 1:end])
                    out[c] = {"code": c, "title": m.group(2).strip(), "pdf_page": n, "text": re.sub(r"[ \t]+", " ", body).strip()}
                    break
        if len(out) == len(codes): return out
    return out

def stage1_fetch(codes=NAICS_CANDIDATES):
    path, accessed = fetch_naics_manual()
    entries = manual_entries(path, codes)
    missing = [c for c in codes if c not in entries]
    if missing:
        raise RuntimeError(f"Codes not found in manual text: {missing}. Paste their definitions from {NAICS_LANDING} into naics/manual_paste.txt.")
    os.makedirs(NAICS_DIR, exist_ok=True)
    record = {"source": "2022 NAICS Manual (U.S. Census Bureau / Executive Office of the President, OMB)", "url": NAICS_MANUAL_URL,
              "landing_page": NAICS_LANDING, "accessed": accessed, "entries": entries}
    json.dump(record, open(os.path.join(NAICS_DIR, "naics_2022_candidates.json"), "w"), indent=2)
    with open(os.path.join(NAICS_DIR, "naics_2022_candidates.md"), "w") as fh:
        fh.write(f"# NAICS 2022 candidate definitions\n\nSource: {NAICS_MANUAL_URL}\nAccessed: {accessed}\n\n")
        for c in codes:
            e = entries[c]; fh.write(f"## {c} {e['title']} (manual PDF page {e['pdf_page']})\n\n{e['text']}\n\n")
    return record

NAICS = stage1_fetch()
print(f"Fetched {len(NAICS['entries'])} candidate entries from the 2022 NAICS Manual (accessed {NAICS['accessed']}):")
for c, e in NAICS["entries"].items():
    n_cross = len(re.findall(r"^[•]", e["text"], re.M))
    print(f"  {c}  {e['title']:<62} manual PDF p.{e['pdf_page']}  ({len(e['text'])} chars, {n_cross} cross-reference bullets)")

Fetched 6 candidate entries from the 2022 NAICS Manual (accessed 2026-09-19):
  541611  Administrative Management and General Management Consulting Services manual PDF p.469  (2274 chars, 7 cross-reference bullets)
  541219  Other Accounting Services                                      manual PDF p.460  (789 chars, 3 cross-reference bullets)
  541211  Offices of Certified Public Accountants                        manual PDF p.459  (1047 chars, 3 cross-reference bullets)
  541618  Other Management Consulting Services                           manual PDF p.471  (1061 chars, 4 cross-reference bullets)
  541990  All Other Professional, Scientific, and Technical Services     manual PDF p.481  (4264 chars, 13 cross-reference bullets)
  561110  Office Administrative Services                                 manual PDF p.487  (2103 chars, 10 cross-reference bullets)


**Choice made:** **541611 Administrative Management and General Management Consulting Services.**

**Closest neighbor rejected:** 541219 *Other Accounting Services* (541211 *Offices of Certified Public Accountants* is the other close
neighbor: it requires accountants "certified to audit the accounting records", i.e. attest work, which a fractional CFO does not sell).

**Honest limit:** this is a *classification argument*, not a market-data argument. Only one of the three downloaded reports covers
management consulting; the two accounting reports describe the neighbor we rejected. The mismatch table below shows this.

In [5]:
_norm = lambda s: re.sub(r"\s+", " ", s).strip().lower()

def stage1_choose(naics, chosen=None, neighbor=None, quotes=None, justification=None):
    """Validate the configured choice against the fetched manual text and return the record used by the brief."""
    chosen, neighbor = chosen or CHOSEN_CODE, neighbor or NEIGHBOR_CODE
    quotes, justification = quotes or JUSTIFICATION_QUOTES, justification or NAICS_JUSTIFICATION
    assert chosen in naics["entries"] and neighbor in naics["entries"], "chosen/neighbor code must be among the fetched candidates"
    for code, qs in quotes.items():
        for q in qs:
            assert _norm(q) in _norm(naics["entries"][code]["text"]), f"Justification quote not found in {code}: {q!r}"
    return {"chosen_code": chosen, "chosen_title": naics["entries"][chosen]["title"], "neighbor_code": neighbor,
            "neighbor_title": naics["entries"][neighbor]["title"], "justification": justification, "manual_url": naics["url"],
            "landing_page": naics["landing_page"], "accessed": naics["accessed"],
            "candidates": [{"code": c, "title": e["title"], "manual_pdf_page": e["pdf_page"]} for c, e in naics["entries"].items()],
            "quotes_verified": sum(len(v) for v in quotes.values())}

NAICS_CHOICE = stage1_choose(NAICS)
print("Chosen NAICS:", NAICS_CHOICE["chosen_code"], "-", NAICS_CHOICE["chosen_title"])
print("\nWhy this and not", NAICS_CHOICE["neighbor_code"] + ":\n" + textwrap.fill(NAICS_CHOICE["justification"], 118))
print("\nAll", NAICS_CHOICE["quotes_verified"], "quoted phrases verified against the fetched manual text.")
print("\nManual's own cross-reference on the neighbor (541219):")
print(textwrap.indent(textwrap.fill(NAICS["entries"]["541219"]["text"].split("Cross-References.")[-1].replace("\n", " ")[:420], 110), "   "))

Chosen NAICS: 541611 - Administrative Management and General Management Consulting Services

Why this and not 541219:
We chose 541611 because the manual defines it as "providing operating advice and assistance to businesses" on issues
such as "financial planning and budgeting" and lists "Financial management (except investment advice) consulting
services" as an example, which describes advice to a client's management; the closest neighbor, 541219, is instead
defined as "providing accounting services (except tax return preparation services only or payroll services only)" for
establishments outside CPA offices, and its own text says "Accountant (except CPA) offices, bookkeeper offices, and
billing offices are included", so it is organized around producing the books rather than advising management on them.

All 5 quoted phrases verified against the fetched manual text.

Manual's own cross-reference on the neighbor (541219):
    • Establishments of CPAs are classified in U.S. Industry 5412

**Which code does each downloaded report cover?** Read from the extraction files (the agent recorded each report's own stated code and scope, and code verifies those quotes in Stage 3).
The result is a mismatch table, not a footnote: only one of three reports sits on our chosen code.

In [6]:
def report_scope_table(extractions, docs, chosen):
    rows = []
    for d, e in extractions.items():
        sc = e["report_scope"]
        rows.append({"doc_id": d, "title": docs[d]["title"], "stated_code": sc["stated_industry_code"], "naics_2022_listed": sc["naics_2022_listed"],
                     "covers_chosen_code": chosen in sc["naics_2022_listed"], "relation": sc["relation_to_chosen_naics"]})
    return rows

# (printed after Stage 2 has loaded the extractions; see the cell after Stage 2)

## Stage 2: Extraction (seven signals, each with page-level evidence)

**Choice:** the extraction schema is defined *first* (`extraction_schema.json`). The default extractor, `claude_code_files`, loads
`extractions/<doc_id>.json`, which **Claude Code (an LLM agent) wrote during the build session** by reading the cached page text of one document at a
time. It is not an API call and no model runs inside this notebook. Every signal carries `evidence[]` = `doc_id, pdf_page, section_heading, quote (<=25 words)`,
a `status` (FOUND / PARTIAL / NOT_FOUND), a `confidence`, and notes on scope.

**Alternatives rejected:** (1) a paid LLM API (the project is zero-cost by requirement); (2) regex/keyword extraction (cannot read
a market-share table or judge "biggest trend"); (3) one pass over all three reports at once (a conclusion from one report would leak into another's extraction,
so each document got its own pass).

**Optional second backend:** `extractor="gemini"` calls Google's Gemini API with a free AI Studio key (`GEMINI_API_KEY`). It is **implemented, untested**:
no key was available in the build session, so it has never called the live API. Its parsing, chunking and rate-limit logic was exercised only against a mock (below).
Its output would go through the same code-side verifier, so a bad Gemini extraction would fail loudly instead of being trusted.

**Limits:** LLM extraction is not deterministic; re-running the extraction step could produce different signals or different quotes.
Verification catches wrong quotes and numbers, not wrong interpretations.

In [7]:
EXTRACTION_SCHEMA = json.load(open("extraction_schema.json"))
SIGNAL_META = {
    "S1": ("Industry size and five-year growth", "context"),
    "S2": ("Top competitors and market share", "rivalry"),
    "S3": ("Regulatory or compliance pressure", "barriers to entry"),
    "S4": ("Key-input concentration or fragility (talent, software)", "supplier power"),
    "S5": ("Customer concentration or fragmentation", "buyer power"),
    "S6": ("Biggest trend of the next five years", "chosen, with reason"),
    "S7": ("Biggest threat of the next five years", "chosen, with reason"),
}

def load_extractions_claude_files(extract_dir=EXTRACT_DIR):
    """Default extractor: read the JSON files the Claude Code agent wrote."""
    out = {}
    for f in sorted(glob.glob(os.path.join(extract_dir, "*.json"))):
        ext = json.load(open(f))
        out[ext["doc_id"]] = ext
    if not out:
        raise FileNotFoundError(f"No extraction files in {extract_dir}/. Use extractor='gemini' (needs GEMINI_API_KEY) or add files.")
    return out

# ---- Gemini backend: IMPLEMENTED, UNTESTED against the live API ------------------------------------------------------------------
GEMINI_API_ROOT = "https://generativelanguage.googleapis.com/v1beta"
GEMINI_MIN_INTERVAL_S = 13          # free tier is roughly 5-10 requests/min; 13 s spacing stays under 5/min
GEMINI_MAX_CHARS_PER_CALL = 350_000 # documents longer than this are split into page chunks and merged
GEMINI_TEST_STATUS = "implemented, untested (never called the live Gemini API; logic exercised only with a mock)"
_last_gemini_call = [0.0]

def gemini_pick_model(api_key, session=requests):
    """Pick a model from the live list (never a hard-coded name). Prefers the newest stable 'flash' model that supports generateContent."""
    r = session.get(f"{GEMINI_API_ROOT}/models", headers={"x-goog-api-key": api_key}, params={"pageSize": 200}, timeout=30)
    r.raise_for_status()
    names = [m["name"].split("/", 1)[1] for m in r.json().get("models", []) if "generateContent" in m.get("supportedGenerationMethods", [])]
    bad = ("lite", "image", "tts", "live", "audio", "embedding", "preview", "exp", "thinking", "vision")
    pool = [n for n in names if "flash" in n and not any(b in n for b in bad)] or [n for n in names if "flash" in n]
    if not pool:
        raise RuntimeError(f"No flash model with generateContent in the live list: {names[:10]}")
    return sorted(pool)[-1]

def gemini_prompt(meta, pages):
    text = "\n".join(f"=== PDF PAGE {p['pdf_page']} ===\n{p['text']}" for p in pages)
    shapes = json.dumps(EXTRACTION_SCHEMA["signal_value_shapes"], indent=1)
    return textwrap.dedent(f"""\
        You are extracting seven industry signals from ONE report, using ONLY the text below. Return a single JSON object that follows this contract.
        doc_id = "{meta['doc_id']}". Keys: doc_id, extracted_by, extraction_date, report_scope, signals[7].
        Each signal: signal_id (S1..S7), force_tag, status (FOUND|PARTIAL|NOT_FOUND), value (object or null), summary (1-2 sentences, own words),
        evidence[] of {{doc_id, pdf_page (integer), section_heading, quote}}, confidence {{level: high|med|low, reason}}, notes.
        Rules: quotes must be copied verbatim from the cited page and be at most 25 words; every number in value must appear in a quote;
        prefer NOT_FOUND to inference; say in notes when a figure covers a broader or narrower industry than fractional CFO services; never use outside knowledge.
        Signals: {json.dumps({k: v[0] for k, v in SIGNAL_META.items()})}
        Value shapes: {shapes}
        report_scope needs stated_industry_code, scope_summary, naics_2022_listed, relation_to_chosen_naics, evidence[].
        TEXT:
        {text}""")

def gemini_call(prompt, api_key, model, session=requests, sleep=time.sleep, clock=time.time, max_tries=5):
    """One rate-limited call. Waits to respect the free-tier spacing, backs off on 429/5xx, returns parsed JSON."""
    for attempt in range(max_tries):
        wait = GEMINI_MIN_INTERVAL_S - (clock() - _last_gemini_call[0])
        if wait > 0: sleep(wait)
        _last_gemini_call[0] = clock()
        r = session.post(f"{GEMINI_API_ROOT}/models/{model}:generateContent", headers={"x-goog-api-key": api_key},
                         json={"contents": [{"parts": [{"text": prompt}]}],
                               "generationConfig": {"responseMimeType": "application/json", "temperature": 0}}, timeout=300)
        if r.status_code in (429, 500, 503):
            sleep(min(120, 2 ** attempt * 15)); continue
        r.raise_for_status()
        txt = r.json()["candidates"][0]["content"]["parts"][0]["text"]
        return json.loads(txt)
    raise RuntimeError("Gemini call failed after retries (rate limit or server error)")

def _chunk_pages(pages, max_chars):
    chunks, cur, n = [], [], 0
    for p in pages:
        if cur and n + len(p["text"]) > max_chars:
            chunks.append(cur); cur, n = [], 0
        cur.append(p); n += len(p["text"])
    if cur: chunks.append(cur)
    return chunks

def merge_chunk_extractions(parts):
    """Merge per-chunk extractions: per signal keep the chunk with the strongest status (FOUND > PARTIAL > NOT_FOUND)."""
    rank = {"FOUND": 2, "PARTIAL": 1, "NOT_FOUND": 0}
    merged = dict(parts[0])
    best = {}
    for part in parts:
        for s in part["signals"]:
            if s["signal_id"] not in best or rank[s["status"]] > rank[best[s["signal_id"]]["status"]]:
                best[s["signal_id"]] = s
    merged["signals"] = [best[k] for k in sorted(best)]
    return merged

def extract_with_gemini(pages_by_doc, docs, api_key, model=None, session=requests, out_dir="extractions_gemini", **kw):
    model = model or GEMINI_MODEL or gemini_pick_model(api_key, session)
    os.makedirs(out_dir, exist_ok=True)
    out = {}
    for doc_id, pages in pages_by_doc.items():
        parts = [gemini_call(gemini_prompt(docs[doc_id], ch), api_key, model, session=session, **kw)
                 for ch in _chunk_pages(pages, GEMINI_MAX_CHARS_PER_CALL)]
        ext = merge_chunk_extractions(parts) if len(parts) > 1 else parts[0]
        ext["doc_id"] = doc_id
        ext["extracted_by"] = f"Gemini model {model} via the free AI Studio API (not Claude); quotes still verified in code"
        ext["extraction_date"] = datetime.date.today().isoformat()
        json.dump(ext, open(os.path.join(out_dir, f"{doc_id}.json"), "w"), indent=1)
        out[doc_id] = ext
    return out

def run_extractor(extractor, pages=None, docs=None):
    if extractor == "claude_code_files":
        return load_extractions_claude_files(), "claude_code_files: Claude Code (LLM agent) wrote these files in the build session"
    if extractor == "gemini":
        key = os.environ.get("GEMINI_API_KEY")
        if not key:
            raise RuntimeError("extractor='gemini' needs GEMINI_API_KEY (free AI Studio key, no billing). Not set, so nothing was run.")
        by_doc = collections.defaultdict(list)
        for p in pages or []: by_doc[p["doc_id"]].append(p)
        return extract_with_gemini(by_doc, docs, key), f"gemini: run at {datetime.datetime.now().isoformat(timespec='seconds')} (status of this backend: {GEMINI_TEST_STATUS})"
    raise ValueError(f"unknown extractor {extractor!r}")

In [8]:
EXTRACTIONS, EXTRACTOR_NOTE = run_extractor(EXTRACTOR, PAGES, DOCS)
print("Extractor:", EXTRACTOR_NOTE, "\n")
print(f"{'doc_id':<36}{'sig':<5}{'status':<10}{'conf':<6}{'#quotes':<8} force_tag")
for d, ext in EXTRACTIONS.items():
    for s in ext["signals"]:
        print(f"{d:<36}{s['signal_id']:<5}{s['status']:<10}{s['confidence']['level']:<6}{len(s['evidence']):<8} {s['force_tag']}")
    print(f"  extracted_by: {ext['extracted_by'][:110]}\n")

Extractor: claude_code_files: Claude Code (LLM agent) wrote these files in the build session 

doc_id                              sig  status    conf  #quotes  force_tag
firstresearch_accounting-services   S1   PARTIAL   med   3        context
firstresearch_accounting-services   S2   PARTIAL   low   2        rivalry
firstresearch_accounting-services   S3   FOUND     med   4        barriers to entry
firstresearch_accounting-services   S4   PARTIAL   low   3        supplier power
firstresearch_accounting-services   S5   PARTIAL   low   3        buyer power
firstresearch_accounting-services   S6   FOUND     med   2        context / substitutes-adjacent (technology)
firstresearch_accounting-services   S7   FOUND     med   2        context (macro demand risk)
  extracted_by: Claude Code (an LLM agent, model claude-sonnet-5) during the build session; not a human, and not a paid API ca

ibisworld_54121c                    S1   FOUND     med   4        context
ibisworld_54121c                

**Gemini backend: offline logic check (mock only).** The cell below feeds the Gemini code path a *fake* HTTP session that returns a canned
response. It checks that model selection from a list, prompt building, rate-limit spacing, a 429 retry, chunk merging and JSON parsing all run without error.
It does **not** show that Gemini follows the schema, that the endpoint or field names are current, or that the free tier accepts this request. That is why
the backend stays labelled *implemented, untested*.

In [9]:
class _FakeResp:
    def __init__(self, code, payload): self.status_code, self._p = code, payload
    def json(self): return self._p
    def raise_for_status(self):
        if self.status_code >= 400: raise RuntimeError(f"HTTP {self.status_code}")

class _FakeSession:
    """Stands in for `requests`; never touches the network."""
    def __init__(self, canned): self.canned, self.calls, self._n = canned, [], 0
    def get(self, url, **kw):
        return _FakeResp(200, {"models": [{"name": "models/gemini-9-pro", "supportedGenerationMethods": ["generateContent"]},
                                          {"name": "models/gemini-9-flash-lite", "supportedGenerationMethods": ["generateContent"]},
                                          {"name": "models/gemini-9-flash", "supportedGenerationMethods": ["generateContent"]},
                                          {"name": "models/gemini-8-flash", "supportedGenerationMethods": ["generateContent"]},
                                          {"name": "models/text-embedding-9", "supportedGenerationMethods": ["embedContent"]}]})
    def post(self, url, **kw):
        self._n += 1; self.calls.append(url)
        if self._n == 1: return _FakeResp(429, {})                       # first call is rate-limited -> must retry
        return _FakeResp(200, {"candidates": [{"content": {"parts": [{"text": json.dumps(self.canned)}]}}]})

_doc = next(iter(EXTRACTIONS))
_fake = _FakeSession(EXTRACTIONS[_doc])
_slept = []
_fake_pages = [{"doc_id": _doc, "pdf_page": i, "text": "x" * 200} for i in range(1, 4)]
GEMINI_MAX_CHARS_PER_CALL, _saved = 500, GEMINI_MAX_CHARS_PER_CALL                # force 2 chunks so the merge path runs
_out = extract_with_gemini({_doc: _fake_pages}, {_doc: {"doc_id": _doc}}, "FAKE-KEY", session=_fake,
                           out_dir=os.path.join(CACHE_DIR, "gemini_mock"), sleep=_slept.append, clock=lambda: 1e9)
GEMINI_MAX_CHARS_PER_CALL = _saved
assert gemini_pick_model("k", _fake) == "gemini-9-flash", "model must come from the live list, newest stable flash"
assert len(_fake.calls) == 3 and all("gemini-9-flash:generateContent" in u for u in _fake.calls), "expected 1 retry + 2 chunk calls"
assert len(_out[_doc]["signals"]) == 7 and _out[_doc]["extracted_by"].startswith("Gemini")
print("Mock check passed: picked model from list, retried after a 429, made", len(_fake.calls) - 1, "chunk calls, merged to 7 signals.")
print("Status of the Gemini backend:", GEMINI_TEST_STATUS)

Mock check passed: picked model from list, retried after a 429, made 2 chunk calls, merged to 7 signals.
Status of the Gemini backend: implemented, untested (never called the live Gemini API; logic exercised only with a mock)


In [10]:
SCOPE_TABLE = report_scope_table(EXTRACTIONS, DOCS, CHOSEN_CODE)
print(f"Chosen code: {CHOSEN_CODE}. Which report covers what:\n")
for r in SCOPE_TABLE:
    print(f"- {r['doc_id']}  |  stated code: {r['stated_code']}  |  NAICS 2022 listed: {r['naics_2022_listed'] or 'none stated'}  |  covers chosen code: {r['covers_chosen_code']}")
    print(textwrap.indent(textwrap.fill(r["relation"], 112), "    "))

Chosen code: 541611. Which report covers what:

- firstresearch_accounting-services  |  stated code: None  |  NAICS 2022 listed: none stated  |  covers chosen code: False
    No code stated, so a match cannot be confirmed. The activities listed (auditing, bookkeeping, payroll, tax)
    describe the accounting family (5412), not management consulting 541611.
- ibisworld_54121c  |  stated code: 54121c  |  NAICS 2022 listed: ['541211', '541219']  |  covers chosen code: False
    Mismatch: covers 541211 and 541219, the accounting neighbors rejected in Stage 1, not 541611. Audit is about
    half of its revenue, which fractional CFO work is not.
- ibisworld_54161  |  stated code: 54161  |  NAICS 2022 listed: ['54161', '541611', '541613', '541614', '541618']  |  covers chosen code: True
    Superset of the chosen 541611 (all management consulting incl. marketing, operations, HR, IT); financial
    management consulting is only one product line.


## Stage 3: Verification, in code (`verify.py`, shown in full below)

**Choice:** the LLM's quotes are never trusted. For every evidence quote, code normalises whitespace, case, hyphens, smart quotes and dashes and checks the
quote appears on the **cited page**; if not, on an **adjacent page** (labelled `verified_adjacent_page`); otherwise a **fuzzy match** (rapidfuzz >= 90) is tried.
Each item ends `verified`, `verified_adjacent_page` or `FAILED`. Quotes over 25 words fail (licensing). Every number in a signal's `value` must appear in that signal's
verified quotes. A signal left with no verified evidence becomes `UNVERIFIED` and goes to the gaps list.

**Alternative rejected:** asking the LLM to "double-check itself". A model that misquoted once can misquote again while confirming; string matching against the page cannot.
**Fuzzy matching was tightened:** a negative control (changing "grew" to "shrank") passed the plain >= 90 rule at 95.4, so the shipped rule also demands that every word and every digit run
of the quote occur in the matched window. Fuzzy matching is for PDF-extraction glitches, not for paraphrase.

**What this proves:** the quote exists on that page. **What it does not prove:** that the quote supports the interpretation in `summary`/`value`. A human still has to read the spot-check list.

In [11]:
#!/usr/bin/env python3
"""Stage 3 - verification, in code, of every quote the LLM agent wrote.

What this proves:   each evidence quote appears (after normalisation) on the page it cites, or on an adjacent page.
What it cannot prove: that the quote means what the extraction says it means. That still needs a human.

Rules implemented (see the notebook's Stage 3 cell for the reasoning):
  * normalise whitespace, case, hyphens (joined), smart quotes, dashes;
  * exact containment on the cited page (checked against BOTH text variants cached at ingest);
  * else exact containment on page-1 / page+1  -> "verified_adjacent_page";
  * else fuzzy match (rapidfuzz partial_ratio >= 90) on the cited page, accepted ONLY if every digit run AND every
    word of the quote also appears in the matched window. So '$157.4' can never fuzzy-match '$158.4', and 'grew' can
    never fuzzy-match 'shrank': fuzzy matching is there for extraction glitches, not for paraphrase.
  * quotes over 25 words, empty quotes and unknown pages FAIL;
  * every number in a signal's value must appear in that signal's verified quotes.

CLI:  python verify.py            run one verification round and append it to verification_log.json (max 2 rounds)
      python verify.py --reset    start the log over (round 1)
"""
import json, re, glob, os, sys, unicodedata, datetime, hashlib
from rapidfuzz import fuzz

FUZZY_MIN = 90
MAX_WORDS = 25
MAX_ROUNDS = 2
STATUSES = {"FOUND", "PARTIAL", "NOT_FOUND"}

# --------------------------------------------------------------------------------------------------------------------
# normalisation
# --------------------------------------------------------------------------------------------------------------------
_SMART = {"‘": "'", "’": "'", "‚": "'", "‛": "'", "“": '"', "”": '"', "„": '"',
          "–": "-", "—": "-", "−": "-", " ": " ", "‑": "-"}

def normalize(s):
    """Lower-case; smart quotes/dashes to ASCII; hyphens joined (so 'well-\\nqualified' == 'well-qualified'); whitespace collapsed."""
    s = unicodedata.normalize("NFKC", s or "")
    s = "".join(c for c in s if unicodedata.category(c) != "Co")   # private-use icon glyphs that IBISWorld PDFs embed in tables
    for a, b in _SMART.items():
        s = s.replace(a, b)
    s = re.sub(r"-\s*", "", s)          # drop hyphens and any line-break whitespace after them
    return re.sub(r"\s+", " ", s).strip().lower()

def numbers_in(text):
    text = re.sub(r"(?<=\d),(?=\d{3})", "", text or "")
    return [float(x) for x in re.findall(r"\d+(?:\.\d+)?", text)]

# --------------------------------------------------------------------------------------------------------------------
# page index
# --------------------------------------------------------------------------------------------------------------------
def build_page_index(pages):
    """pages: list of dicts with doc_id, pdf_page, printed_page, text, text_raw -> {(doc, page): {'variants': [...], 'printed': x}}"""
    idx = {}
    for p in pages:
        variants = [normalize(p.get("text", "")), normalize(p.get("text_raw", ""))]
        idx[(p["doc_id"], p["pdf_page"])] = {"variants": [v for v in variants if v], "printed": p.get("printed_page")}
    return idx

def _same_tokens(nq, window):
    """Guard for fuzzy matches: digit runs must agree exactly, and every word of the quote must occur in the window."""
    if sorted(re.findall(r"\d+", nq)) != sorted(re.findall(r"\d+", window)):
        return False
    have = set(re.findall(r"[a-z0-9$%.]+", window))
    return all(t in have for t in re.findall(r"[a-z0-9$%.]+", nq))

def _contains(variants, nq):
    return any(nq in v for v in variants)

def _fuzzy(variants, nq):
    best_score, best_window = 0.0, ""
    for v in variants:
        al = fuzz.partial_ratio_alignment(nq, v)
        if al is not None and al.score > best_score:
            best_score, best_window = al.score, v[al.dest_start:al.dest_end]
    return best_score, best_window

def check_quote(idx, doc_id, page, quote):
    """Return dict(result, method, score, matched_page, printed_page, reason)."""
    out = {"result": "FAILED", "method": None, "score": None, "matched_page": None, "printed_page": None, "reason": None}
    here = idx.get((doc_id, page))
    if here is None:
        out["reason"] = "cited page does not exist in the cache"; return out
    out["printed_page"] = here["printed"]
    words = len((quote or "").split())
    if not (quote or "").strip():
        out["reason"] = "empty quote"; return out
    if words > MAX_WORDS:
        out["reason"] = f"quote has {words} words (limit {MAX_WORDS})"; return out
    nq = normalize(quote)
    if _contains(here["variants"], nq):
        out.update(result="verified", method="exact", score=100.0, matched_page=page); return out
    for adj in (page - 1, page + 1):
        other = idx.get((doc_id, adj))
        if other and _contains(other["variants"], nq):
            out.update(result="verified_adjacent_page", method="exact", score=100.0, matched_page=adj,
                       reason=f"quote is on PDF p.{adj}, not the cited p.{page}"); return out
    score, window = _fuzzy(here["variants"], nq)
    out["score"] = round(score, 1)
    if score >= FUZZY_MIN and _same_tokens(nq, window):
        out.update(result="verified", method="fuzzy", matched_page=page); return out
    for adj in (page - 1, page + 1):
        other = idx.get((doc_id, adj))
        if other:
            s2, w2 = _fuzzy(other["variants"], nq)
            if s2 >= FUZZY_MIN and _same_tokens(nq, w2):
                out.update(result="verified_adjacent_page", method="fuzzy", score=round(s2, 1), matched_page=adj,
                           reason=f"fuzzy match on PDF p.{adj}"); return out
    out["reason"] = f"not found on p.{page} or adjacent pages (best fuzzy score {out['score']})"
    return out

# --------------------------------------------------------------------------------------------------------------------
# structure + number checks
# --------------------------------------------------------------------------------------------------------------------
def schema_errors(ext, schema_path="extraction_schema.json"):
    errs = []
    try:
        import jsonschema
        schema = json.load(open(schema_path))
        v = jsonschema.Draft7Validator(schema)
        errs += [f"{'/'.join(map(str, e.path)) or '<root>'}: {e.message[:140]}" for e in v.iter_errors(ext)]
    except ImportError:
        for k in ("doc_id", "extracted_by", "report_scope", "signals"):
            if k not in ext: errs.append(f"missing key {k}")
    ids = [s.get("signal_id") for s in ext.get("signals", [])]
    if sorted(ids) != [f"S{i}" for i in range(1, 8)]:
        errs.append(f"signals must be exactly S1..S7, got {ids}")
    for s in ext.get("signals", []):
        for e in s.get("evidence", []):
            if e.get("doc_id") != ext.get("doc_id"):
                errs.append(f"{s.get('signal_id')}: evidence doc_id {e.get('doc_id')} differs from file doc_id")
    return errs

def numeric_leaves(value, path=""):
    """Numbers to be matched against quotes: numeric leaves, plus every number inside strings under keys ending '_range'."""
    found = []
    if isinstance(value, bool) or value is None:
        return found
    if isinstance(value, (int, float)):
        return [(path, float(value))]
    if isinstance(value, dict):
        for k, v in value.items():
            if isinstance(v, str) and k.endswith("_range"):
                found += [(f"{path}/{k}", n) for n in numbers_in(v)]
            else:
                found += numeric_leaves(v, f"{path}/{k}")
    elif isinstance(value, list):
        for i, v in enumerate(value):
            found += numeric_leaves(v, f"{path}[{i}]")
    return found

def check_numbers(signal, verified_quotes):
    """Every numeric leaf of the value must appear in the union of the signal's VERIFIED quotes."""
    have = set()
    for q in verified_quotes:
        have.update(round(n, 6) for n in numbers_in(q))
    rows = []
    for path, n in numeric_leaves(signal.get("value")):
        rows.append({"path": path, "number": n, "in_quote": round(n, 6) in have})
    return rows

# --------------------------------------------------------------------------------------------------------------------
# verification driver
# --------------------------------------------------------------------------------------------------------------------
def load_extractions(extract_dir="extractions"):
    out = {}
    for f in sorted(glob.glob(os.path.join(extract_dir, "*.json"))):
        ext = json.load(open(f))
        out[ext.get("doc_id", os.path.basename(f)[:-5])] = ext
    return out

def verify_all(extractions, idx):
    """Check every evidence quote (signals + report_scope). Returns (items, number_rows, structure_errors)."""
    items, number_rows, struct = [], [], {}
    for doc_id, ext in extractions.items():
        struct[doc_id] = schema_errors(ext)
        groups = [("SCOPE", ext["report_scope"].get("evidence", []))] + [(s["signal_id"], s.get("evidence", [])) for s in ext["signals"]]
        for sid, evs in groups:
            for i, ev in enumerate(evs):
                r = check_quote(idx, ev.get("doc_id", doc_id), ev.get("pdf_page", -1), ev.get("quote", ""))
                items.append({"doc_id": doc_id, "signal_id": sid, "evidence_index": i, "pdf_page": ev.get("pdf_page"),
                              "section_heading": ev.get("section_heading"), "quote": ev.get("quote"), **r})
        for s in ext["signals"]:
            if s["status"] == "NOT_FOUND":
                continue
            good = [it["quote"] for it in items if it["doc_id"] == doc_id and it["signal_id"] == s["signal_id"]
                    and it["result"] in ("verified", "verified_adjacent_page")]
            for row in check_numbers(s, good):
                number_rows.append({"doc_id": doc_id, "signal_id": s["signal_id"], **row})
    return items, number_rows, struct

def summarize(items, number_rows, struct):
    n = len(items)
    passed = sum(1 for it in items if it["result"] in ("verified", "verified_adjacent_page"))
    return {"quotes_checked": n, "passed": passed, "failed": n - passed,
            "passed_exact": sum(1 for it in items if it["result"] == "verified" and it["method"] == "exact"),
            "passed_fuzzy": sum(1 for it in items if it["method"] == "fuzzy" and it["result"] != "FAILED"),
            "passed_adjacent_page": sum(1 for it in items if it["result"] == "verified_adjacent_page"),
            "numbers_checked": len(number_rows), "numbers_unmatched": sum(1 for r in number_rows if not r["in_quote"]),
            "structure_errors": sum(len(v) for v in struct.values())}

def apply_verification(extractions, items, number_rows):
    """Return per-signal verified view. Failed evidence is dropped from support; a FOUND/PARTIAL signal left with no verified
    evidence becomes UNVERIFIED (and is reported as a gap). Unmatched numbers are flagged, not silently kept."""
    view = {}
    for doc_id, ext in extractions.items():
        sigs = {}
        for s in ext["signals"]:
            mine = [it for it in items if it["doc_id"] == doc_id and it["signal_id"] == s["signal_id"]]
            keep = [it for it in mine if it["result"] in ("verified", "verified_adjacent_page")]
            dropped = [it for it in mine if it["result"] == "FAILED"]
            bad_nums = [r for r in number_rows if r["doc_id"] == doc_id and r["signal_id"] == s["signal_id"] and not r["in_quote"]]
            status = s["status"]
            if status in ("FOUND", "PARTIAL") and not keep:
                status = "UNVERIFIED"
            sigs[s["signal_id"]] = {"status_final": status, "status_claimed": s["status"], "verified_evidence": keep,
                                    "dropped_evidence": dropped, "numbers_unmatched": bad_nums}
        view[doc_id] = sigs
    return view

def load_log(path="verification_log.json"):
    return json.load(open(path)) if os.path.exists(path) else None

def write_log(path, items, number_rows, struct, rounds, generated=None):
    log = {"generated": generated or datetime.datetime.now().isoformat(timespec="seconds"),
           "note": "Quotes are <=25 words each. No full page text is stored here. 'verified' means the quote exists on the cited page; it does not mean the interpretation is right.",
           "rounds": rounds, "summary": summarize(items, number_rows, struct), "structure_errors": struct,
           "items": items, "number_checks": number_rows}
    json.dump(log, open(path, "w"), indent=1, ensure_ascii=False)
    return log

def corrected_in_retry(rounds_items):
    """Evidence that FAILED in an earlier round and passes in the last one, matched by (doc, signal, evidence_index)."""
    if len(rounds_items) < 2: return 0
    key = lambda it: (it["doc_id"], it["signal_id"], it["evidence_index"])
    ok = lambda it: it["result"] in ("verified", "verified_adjacent_page")
    first_failed = {key(it) for it in rounds_items[0] if not ok(it)}
    last_ok = {key(it) for it in rounds_items[-1] if ok(it)}
    return len(first_failed & last_ok)


In [12]:
def stage3_verify(extractions, cache_dir=CACHE_DIR, log_path="verification_log.json"):
    """Live mode: re-verify every quote against the cached pages. Replay mode (no PDFs on this machine): reuse the committed log and say so."""
    prev = load_log(log_path)
    rounds = (prev or {}).get("rounds", [])
    if os.path.exists(os.path.join(cache_dir, "pages.jsonl")):
        idx = build_page_index(load_pages_cache(cache_dir))
        items, number_rows, struct = verify_all(extractions, idx)
        log = write_log(log_path, items, number_rows, struct, rounds)
        return items, number_rows, struct, log, "LIVE: every quote re-checked against the PDF page text just extracted"
    if prev is None:
        raise RuntimeError("No PDFs/cache and no committed verification_log.json: nothing to verify against.")
    return prev["items"], prev["number_checks"], prev.get("structure_errors", {}), prev, \
        "REPLAY: PDFs are not on this machine, so quotes were NOT re-checked; showing the committed log from the build session"

ITEMS, NUMROWS, STRUCT, LOG, VERIFY_MODE = stage3_verify(EXTRACTIONS)
SUMM = summarize(ITEMS, NUMROWS, STRUCT)
_prev_rounds = LOG.get("rounds", [])
_failed_r1 = {tuple(k) for k in (_prev_rounds[0]["failed_keys"] if _prev_rounds else [])}
_ok_now = {(i["doc_id"], i["signal_id"], i["evidence_index"]) for i in ITEMS if i["result"] != "FAILED"}
SUMM["corrected_in_retry"] = len(_failed_r1 & _ok_now)
SUMM["rounds_run"] = len(_prev_rounds)
print("Mode:", VERIFY_MODE)
print(f"\nQuotes checked: {SUMM['quotes_checked']} | passed: {SUMM['passed']} (exact {SUMM['passed_exact']}, adjacent page {SUMM['passed_adjacent_page']}, fuzzy {SUMM['passed_fuzzy']})"
      f" | failed: {SUMM['failed']} | corrected in retry: {SUMM['corrected_in_retry']} | fix-and-retry rounds run: {SUMM['rounds_run']} (max {MAX_ROUNDS})")
print(f"Numbers in signal values checked against their quotes: {SUMM['numbers_checked']} | not found: {SUMM['numbers_unmatched']} | schema/structure errors: {SUMM['structure_errors']}")
for it in ITEMS:
    if it["result"] == "FAILED": print("  FAILED:", it["doc_id"], it["signal_id"], it["reason"])
VERIFIED = apply_verification(EXTRACTIONS, ITEMS, NUMROWS)
print("\nSignals whose final status differs from what the extractor claimed:",
      [(d, s) for d, sigs in VERIFIED.items() for s, v in sigs.items() if v["status_final"] != v["status_claimed"]] or "none")
print("\nShort sample of verified evidence (document, page, quote):")
for it in [i for i in ITEMS if i["signal_id"] == "S2"][:3]:
    print(f"  [{it['doc_id']}, PDF p.{it['pdf_page']} (printed {it['printed_page']})] {it['result']}: \"{it['quote'][:90]}\"")

Mode: LIVE: every quote re-checked against the PDF page text just extracted

Quotes checked: 79 | passed: 79 (exact 79, adjacent page 0, fuzzy 0) | failed: 0 | corrected in retry: 0 | fix-and-retry rounds run: 1 (max 2)
Numbers in signal values checked against their quotes: 55 | not found: 0 | schema/structure errors: 0

Signals whose final status differs from what the extractor claimed: none

Short sample of verified evidence (document, page, quote):
  [firstresearch_accounting-services, PDF p.1 (printed 1)] verified: "Leading companies include ADP, H&R Block, and Paychex (all based in the US)."
  [firstresearch_accounting-services, PDF p.5 (printed 5)] verified: "the 50 largest US companies account for just less than 50% of revenue"
  [ibisworld_54121c, PDF p.32 (printed 29)] verified: "Industry-specific company revenue as a share of total industry revenue."


**Verifier self-test.** All real quotes passing is only reassuring if the verifier can also *fail*. This cell takes verified quotes, corrupts them in four ways (change a digit, swap a word,
cite the wrong page, replace with an invented sentence) and checks the verifier rejects every corruption while still accepting the untouched originals. It is a test of the verifier, **not** a finding about the extraction:
in this run the LLM's real quotes had no failures to catch.

In [13]:
def verifier_self_test(items, cache_dir=CACHE_DIR, n=24, seed=7):
    if not os.path.exists(os.path.join(cache_dir, "pages.jsonl")):
        return None
    idx = build_page_index(load_pages_cache(cache_dir))
    rng = random.Random(seed)
    pool = [i for i in items if i["result"] == "verified" and i["signal_id"] != "SCOPE"]
    rng.shuffle(pool)
    rows, seen = collections.Counter(), collections.Counter()
    def bump(kind, caught):
        seen[kind] += 1; rows[kind] += int(caught)
    for it in pool[:n]:
        q, d, p = it["quote"], it["doc_id"], it["pdf_page"]
        bump("control (untouched)", check_quote(idx, d, p, q)["result"] != "FAILED")
        m = re.search(r"\d", q)
        if m:
            bad = q[:m.start()] + str((int(q[m.start()]) + 1) % 10) + q[m.start() + 1:]
            bump("digit changed", check_quote(idx, d, p, bad)["result"] == "FAILED")
        words = [w for w in re.findall(r"[A-Za-z]{5,}", q)]
        if words:
            bump("word swapped", check_quote(idx, d, p, q.replace(rng.choice(words), "zebra", 1))["result"] == "FAILED")
        bump("wrong page (+7)", check_quote(idx, d, p + 7, q)["result"] == "FAILED")
        bump("invented sentence", check_quote(idx, d, p, "Fractional CFO retainers average four thousand dollars per month")["result"] == "FAILED")
    return seen, rows

_st = verifier_self_test(ITEMS)
if _st is None:
    print("Self-test skipped (no page cache on this machine).")
else:
    seen, ok = _st
    print(f"{'test':<24}{'cases':<8}{'as expected':<12}")
    for k in seen: print(f"{k:<24}{seen[k]:<8}{ok[k]:<12}")
    assert all(ok[k] == seen[k] for k in seen), "verifier failed its own self-test"
    print("\nVerifier behaves as intended on all cases (controls accepted, every corruption rejected).")

test                    cases   as expected 
control (untouched)     24      24          
word swapped            24      24          
wrong page (+7)         24      24          
invented sentence       24      24          
digit changed           11      11          

Verifier behaves as intended on all cases (controls accepted, every corruption rejected).


## Stage 4: Reconciliation across documents

**Choice:** for each signal, code gathers every document's *verified* finding and picks one **primary** by a stated rule:
(1) the report whose stated NAICS scope contains the chosen code (541611) beats one that does not; (2) then FOUND beats PARTIAL; (3) then the more recent publication date.
The primary is a *presentation choice*. Nothing is dropped or averaged: any place where two documents (or one document against itself) give different values for the
same metric goes into `conflicts[]` with both sides, page citations and a note on why they differ.

**Alternatives rejected:** averaging or weighting values (it would invent a number no report published, and the reports measure different industries); always preferring the newest report
(the newest report is not the one nearest our NAICS code); silently dropping the loser.

**Limit:** the rule ranks by *scope proximity*, and scope proximity here is weak: the "closest" report (54161) is all management consulting. Different-industry figures are shown as "not comparable", not as disagreement.

In [14]:
PRIMARY_RULE = ("Primary = the document whose stated NAICS scope includes the chosen code; then FOUND before PARTIAL; then the more recent publication date. "
                "Other documents' findings are always shown alongside; conflicting values are never averaged.")

def parse_pub_date(s):
    try: return datetime.datetime.strptime(s.replace("Data Published: ", "").strip(" ."), "%B %Y").date()
    except Exception: return datetime.date.min

def scope_rank(ext, chosen=None):
    chosen = chosen or CHOSEN_CODE
    codes = ext["report_scope"].get("naics_2022_listed", [])
    if chosen in codes: return 0
    if any(c[:5] == chosen[:5] for c in codes): return 1
    return 2

def industry_family(doc, ext):
    codes = ext["report_scope"].get("naics_2022_listed", [])
    if any(c.startswith("5412") for c in codes) or "accounting" in doc.get("title", "").lower(): return "accounting"
    if any(c.startswith("5416") for c in codes): return "management consulting"
    return "other"

def _cite(items, doc_id, sid, prefer_number=None):
    ev = [i for i in items if i["doc_id"] == doc_id and i["signal_id"] == sid and i["result"] in ("verified", "verified_adjacent_page")]
    if prefer_number is not None:
        for i in ev:
            if round(float(prefer_number), 6) in {round(n, 6) for n in numbers_in(i["quote"])}: return _fmt_cite(i)
    return _fmt_cite(ev[0]) if ev else None

def _fmt_cite(i):
    return {"doc_id": i["doc_id"], "section_heading": i["section_heading"], "pdf_page": i["pdf_page"], "printed_page": i["printed_page"],
            "quote": i["quote"], "verification": i["result"]}

def _lead_rating(text):
    m = re.match(r"\s*(Low|Moderate|High)\b", text or "", re.I)
    return m.group(1).capitalize() if m else None

def stage4_reconcile(extractions, verified, items, docs):
    """Returns per-signal records (primary + others + conflicts). Pure code, no model call."""
    order = sorted(extractions, key=lambda d: (scope_rank(extractions[d]), -parse_pub_date(docs[d]["publication_date"] or "").toordinal()))
    records = {}
    for sid in SIGNAL_META:
        cands = []
        for d in extractions:
            sig = next(s for s in extractions[d]["signals"] if s["signal_id"] == sid)
            v = verified[d][sid]
            if v["status_final"] in ("FOUND", "PARTIAL"):
                cands.append((scope_rank(extractions[d]), 0 if v["status_final"] == "FOUND" else 1,
                              -parse_pub_date(docs[d]["publication_date"] or "").toordinal(), d, sig, v))
        cands.sort(key=lambda c: c[:4])
        def pack(c):
            _, _, _, d, sig, v = c
            return {"doc_id": d, "status": v["status_final"], "force_tag": sig["force_tag"], "value": sig["value"], "summary": sig["summary"],
                    "confidence": sig["confidence"], "notes": sig["notes"],
                    "citations": [_fmt_cite(i) for i in v["verified_evidence"]], "numbers_unmatched": v["numbers_unmatched"]}
        records[sid] = {"signal_id": sid, "name": SIGNAL_META[sid][0], "primary": pack(cands[0]) if cands else None,
                        "other_documents": [pack(c) for c in cands[1:]], "conflicts": [], "not_supported_by": [
                            {"doc_id": d, "status": verified[d][sid]["status_final"]} for d in extractions if verified[d][sid]["status_final"] not in ("FOUND", "PARTIAL")]}
    fam = {d: industry_family(docs[d], extractions[d]) for d in extractions}
    val = lambda d, sid: next(s for s in extractions[d]["signals"] if s["signal_id"] == sid)["value"]
    ok = lambda d, sid: verified[d][sid]["status_final"] in ("FOUND", "PARTIAL")

    # (a) same metric, same industry family, different values (S1)
    for metric, label in (("market_size", "Market size (USD bn)"), ("cagr_historical_pct", "Historical 5-yr CAGR (%)"), ("cagr_forecast_pct", "Forecast 5-yr CAGR (%)")):
        docs_with = [d for d in extractions if ok(d, "S1") and val(d, "S1").get(metric) is not None]
        for i, a in enumerate(docs_with):
            for b in docs_with[i + 1:]:
                if fam[a] == fam[b] and val(a, "S1")[metric] != val(b, "S1")[metric]:
                    period = lambda d: val(d, "S1").get("forecast_period" if "forecast" in metric else "historical_period", val(d, "S1").get("market_size_year"))
                    records["S1"]["conflicts"].append({
                        "kind": "value_disagreement_same_industry_family", "metric": label,
                        "sides": [{"doc_id": d, "value": val(d, "S1")[metric], "period_or_year": period(d), "citation": _cite(items, d, "S1", val(d, "S1")[metric])} for d in (a, b)],
                        "why_they_may_differ": "Both describe accounting services but with different scope (First Research includes payroll and tax preparation), vintage (July 2025 vs May 2026) and, for growth, "
                                               "different periods and price basis. The reports do not say which is right."})
    # (b) a document disagreeing with itself
    for d in extractions:
        for sid in SIGNAL_META:
            v = val(d, sid) if ok(d, sid) else None
            if v and "internal_conflict" in v:
                ic = v["internal_conflict"]
                for metric, label in (("market_size", "Market size (USD bn)"), ("cagr_historical_pct", "Historical 5-yr CAGR (%)"), ("cagr_forecast_pct", "Forecast 5-yr CAGR (%)")):
                    if metric in ic and ic[metric] != v.get(metric):
                        records[sid]["conflicts"].append({
                            "kind": "within_document", "metric": label,
                            "sides": [{"doc_id": d, "value": v[metric], "period_or_year": "At a Glance / Performance Snapshot", "citation": _cite(items, d, sid, v[metric])},
                                      {"doc_id": d, "value": ic[metric], "period_or_year": "Executive Summary text", "citation": _cite(items, d, sid, ic[metric])}],
                            "why_they_may_differ": ic.get("where", "")})
    # (c) rating disagreements (different scopes): regulation, entry barriers, buyer power
    for sid, key, label in (("S3", "regulation_level", "Regulation & policy rating"), ("S3", "barriers_to_entry_level", "Barriers to entry rating"),
                            ("S5", "buyer_power_rating", "Buyer power rating")):
        rated = {d: _lead_rating(val(d, sid).get(key)) for d in extractions if ok(d, sid) and val(d, sid).get(key)}
        if len(set(rated.values())) > 1:
            records[sid]["conflicts"].append({
                "kind": "rating_disagreement_different_scope", "metric": label,
                "sides": [{"doc_id": d, "value": val(d, sid)[key], "period_or_year": f"{docs[d]['publisher'].split(' (')[0]} rating/wording", "citation": _cite(items, d, sid)} for d in rated],
                "why_they_may_differ": "Each rating is for a different industry (management consulting vs accounting firms) and the publishers word them differently, so they are not the same measurement; shown side by side, not merged."})
    # (d) same company, different share basis (S2)
    shares = collections.defaultdict(list)
    for d in extractions:
        if ok(d, "S2"):
            for c in val(d, "S2")["competitors"]:
                if c.get("share_pct") is not None or c.get("share_range"):
                    shares[c["name"].lower().replace(" plc", "")].append((d, c))
    for name, lst in shares.items():
        if len({x[0] for x in lst}) > 1:
            records["S2"]["conflicts"].append({
                "kind": "same_firm_different_share_basis", "metric": f"Market share of {lst[0][1]['name']}",
                "sides": [{"doc_id": d, "value": c["share_pct"] if c.get("share_pct") is not None else c["share_range"], "period_or_year": c["share_basis"],
                           "citation": _cite(items, d, "S2", c.get("share_pct"))} for d, c in lst],
                "why_they_may_differ": "The denominators are different industries (accounting vs management consulting), so the shares are not comparable and must not be averaged or added."})
    return records, fam

RECORDS, FAMILY = stage4_reconcile(EXTRACTIONS, VERIFIED, ITEMS, DOCS)
print("Rule:", PRIMARY_RULE, "\n")
print("Industry family per document:", FAMILY, "\n")
for sid, r in RECORDS.items():
    p = r["primary"]
    print(f"{sid} {r['name']}\n   primary: {p['doc_id'] if p else 'NONE'} ({p['status'] if p else '-'}) | other documents: {[o['doc_id'] for o in r['other_documents']]} | conflicts: {len(r['conflicts'])}")
print("\nConflicts kept side by side (never averaged):")
for sid, r in RECORDS.items():
    for c in r["conflicts"]:
        sides = "  vs  ".join(f"{s['doc_id']}={s['value']}" for s in c["sides"])
        print(f"  [{sid}] {c['kind']}: {c['metric']}: {sides}")

Rule: Primary = the document whose stated NAICS scope includes the chosen code; then FOUND before PARTIAL; then the more recent publication date. Other documents' findings are always shown alongside; conflicting values are never averaged. 

Industry family per document: {'firstresearch_accounting-services': 'accounting', 'ibisworld_54121c': 'accounting', 'ibisworld_54161': 'management consulting'} 

S1 Industry size and five-year growth
   primary: ibisworld_54161 (FOUND) | other documents: ['ibisworld_54121c', 'firstresearch_accounting-services'] | conflicts: 5
S2 Top competitors and market share
   primary: ibisworld_54161 (FOUND) | other documents: ['ibisworld_54121c', 'firstresearch_accounting-services'] | conflicts: 4
S3 Regulatory or compliance pressure
   primary: ibisworld_54161 (FOUND) | other documents: ['ibisworld_54121c', 'firstresearch_accounting-services'] | conflicts: 1
S4 Key-input concentration or fragility (talent, software)
   primary: ibisworld_54161 (FOUND) | other

## Stage 5: Gaps (what the pipeline could not find, and what would find it)

**Choice:** gaps are generated by code from the run itself: every (document, signal) that is NOT_FOUND, PARTIAL, UNVERIFIED or low-confidence, every signal no document supports,
every unresolved conflict, plus *structural* gaps that hold whatever the extraction says (there is no industry code for fractional CFO; no report gives a niche share).
Each gap says what is missing and names a *type* of source that could fill it. **Those suggested sources were not consulted**; they come from general knowledge of where such data lives, not from the three reports.

**Alternative rejected:** a generic "limitations" paragraph. It would read the same whatever the run found, and it would not tell a founder which number to go and get.

In [15]:
SIGNAL_SOURCES = {
    "S1": "A report on a closer proxy than all management consulting or all accounting. Census Bureau Economic Census / Nonemployer Statistics receipts for NAICS 541611 and 541219 (census.gov/naics for the code, data.census.gov for the tables), and IBISWorld 'financial management consulting' segment data.",
    "S2": "No report here gives a niche share. Look for a vendor report on outsourced / virtual / fractional CFO or 'bookkeeping and advisory' services (existence not confirmed; not searched), Census Economic Census concentration ratios (top-4/8/20/50 share) by NAICS, or bottom-up counts from state CPA-society firm directories and LinkedIn / Google Business listings.",
    "S3": "Primary law, not industry reports: each target state's Board of Accountancy rules on who may use 'CPA' and offer attest vs non-attest services; AICPA independence rules; state consumer-privacy statutes. Ask a licensed CPA or attorney whether fractional CFO advice needs a licence.",
    "S4": "BLS Occupational Employment and Wage Statistics for financial managers and accountants by state; accounting-software vendor market-share studies (QuickBooks, Xero, NetSuite class); AICPA firm-staffing surveys on CPA supply.",
    "S5": "Census Statistics of U.S. Businesses (SUSB) receipts-size tables and SBA Office of Advocacy small-business profiles for the $2M-$20M revenue band; a buyer survey (owners' willingness to pay for part-time finance leadership).",
    "S6": "Primary research: interviews with 5-10 fractional CFOs and 5-10 SMB owners; AICPA / accounting-trade surveys on advisory demand.",
    "S7": "Primary research plus a review of platform-based competitors' public pricing and positioning (not in these reports).",
}

def stage5_gaps(extractions, verified, records, docs, naics=None, chosen=None):
    naics, chosen = naics or NAICS, chosen or CHOSEN_CODE
    gaps = []
    for d, ext in extractions.items():
        for s in ext["signals"]:
            v = verified[d][s["signal_id"]]
            reasons = []
            if v["status_final"] != "FOUND": reasons.append(v["status_final"])
            if s["confidence"]["level"] == "low": reasons.append("low confidence")
            if v["numbers_unmatched"]: reasons.append("numbers not in quotes")
            if reasons:
                gaps.append({"scope": "document-signal", "signal_id": s["signal_id"], "doc_id": d, "why": " + ".join(reasons),
                             "what_is_missing": s["notes"], "what_would_find_it": SIGNAL_SOURCES[s["signal_id"]]})
    for sid, r in records.items():
        if r["primary"] is None:
            gaps.append({"scope": "signal", "signal_id": sid, "doc_id": None, "why": "no document supports this signal",
                         "what_is_missing": "Every report returned NOT_FOUND or UNVERIFIED.", "what_would_find_it": SIGNAL_SOURCES[sid]})
        for c in r["conflicts"]:
            if c["kind"] not in ("value_disagreement_same_industry_family", "within_document"):
                continue        # rating and share-basis differences are explained by scope (kept in the brief), not unresolved
            gaps.append({"scope": "conflict", "signal_id": sid, "doc_id": None, "why": f"unresolved conflict: {c['metric']}",
                         "what_is_missing": "Values differ: " + "; ".join(f"{s['doc_id']} = {s['value']}" for s in c["sides"]) + ". " + c["why_they_may_differ"],
                         "what_would_find_it": "The publisher's methodology note (IBISWorld / First Research analyst support) or a primary statistical source such as the Census Economic Census."})
    exts = extractions
    rel = "; ".join(f"{d}: {e['report_scope']['relation_to_chosen_naics']}" for d, e in exts.items())
    dates = ", ".join(f"{d} = {docs[d]['publication_date']}" for d in exts)
    structural = [
        ("No industry code for fractional CFO", f"Fractional CFO has no NAICS code, so every number in this brief describes a PROXY industry (chosen: {chosen}, "
         f"{naics['entries'][chosen]['title']}). Growth, size and share for the fractional niche itself are unknown, and the proxy may grow at a different rate.",
         "Nothing will publish this directly; a bottom-up estimate (number of $2M-$20M firms x adoption rate x average retainer) built from Census SUSB plus buyer interviews."),
        ("No report gives market share for the fractional niche", "The only shares printed are for the Big Four and a few global firms (of management consulting or of CPA accounting); First Research prints none. "
         "None of those firms is a competitor for $2M-$20M clients in the way a fractional CFO boutique or platform would be.", SIGNAL_SOURCES["S2"]),
        ("Reports do not line up with the chosen code", rel, "Buy or access a report whose stated code matches the chosen NAICS, or one on a niche named 'outsourced accounting/CFO services'."),
        ("Mixed and dated vintages", f"Publication dates: {dates}. The First Research printout also embeds valuation multiples last updated January 2023 and an economic indicator dated May 2026 inside a report whose data was published July 2025.",
         "Re-pull each report at analysis time and record the publication date next to each number."),
        ("No pricing, retainer or margin data for the niche", "None of the seven signals covers what fractional CFOs charge or earn; the IBISWorld profit margins are for whole industries.",
         "Interviews and public rate cards; AICPA / trade compensation surveys."),
        ("Verification proves quotes exist, not that they were understood", "Code confirmed each quote is on its cited page. It cannot confirm that the LLM's summary, its choice of 'biggest' trend/threat, or its reading of a table row is right.",
         "A human reading the spot-check list against the PDFs (see the method section)."),
    ]
    for title, missing, find in structural:
        gaps.append({"scope": "structural", "signal_id": None, "doc_id": None, "why": title, "what_is_missing": missing, "what_would_find_it": find})
    return gaps

GAPS = stage5_gaps(EXTRACTIONS, VERIFIED, RECORDS, DOCS)
by_scope = collections.Counter(g["scope"] for g in GAPS)
print("Gaps:", dict(by_scope), "| total", len(GAPS), "\n")
for g in GAPS:
    if g["scope"] in ("document-signal", "signal"):
        print(f"[{g['signal_id']}] {g['doc_id']}: {g['why']}\n     missing: {textwrap.shorten(g['what_is_missing'], 150)}")
print("\nStructural gaps:")
for g in GAPS:
    if g["scope"] == "structural": print(" -", g["why"])

Gaps: {'document-signal': 4, 'conflict': 5, 'structural': 6} | total 15 

[S1] firstresearch_accounting-services: PARTIAL
     missing: Broader industry (accounting, tax prep, bookkeeping, payroll). The report was published July 2025, older than both IBISWorld reports.
[S2] firstresearch_accounting-services: PARTIAL + low confidence
     missing: No competitor share percentages, so this signal cannot fill the 'top 3-5 with shares' requirement. Leaders named are payroll and tax-prep firms, [...]
[S4] firstresearch_accounting-services: PARTIAL + low confidence
     missing: Supplier power as a concept is not analysed in this report; only the dependence on people and software is stated.
[S5] firstresearch_accounting-services: PARTIAL + low confidence
     missing: The only small-business statement is that owners rely on accountants for advice, which supports demand for advisory work but does not quantify buyers.

Structural gaps:
 - No industry code for fractional CFO
 - No report gives m

## Brief generation (`brief.json`, `brief.md`)

**Choice:** the brief is *assembled from the reconciled, verified records*, never re-written by a model. Each signal shows the primary finding, every other document's finding,
citations as `[doc_id, section, PDF page (printed page)]` with the short verbatim quote and its verification label, and any conflict side by side. Five (document, signal) pairs are drawn at random (fixed,
recorded seed) into a **spot-check list** for a human to compare against the PDFs.

**Alternative rejected:** a fluent narrative summary. It would hide which sentence came from which page and would be the one part of the pipeline nobody could verify.
**Limit:** the brief inherits every weakness of the proxy industry described in the gaps.

In [16]:
SPOT_SEED = 20260919

def fmt_value(v, depth=0):
    """Compact one-line rendering of a signal value for the markdown/PDF brief."""
    if isinstance(v, dict) and "competitors" in v:
        def one(c):
            share = f"{c['share_pct']}%" if c.get("share_pct") is not None else (f"band {c['share_range']}%" if c.get("share_range") else "no share given")
            return f"{c['name']} {share}" + (f" (revenue ${c['revenue_usd_bn']}bn)" if c.get("revenue_usd_bn") is not None else "")
        bases = list(dict.fromkeys(c["share_basis"] for c in v["competitors"]))
        return "; ".join(one(c) for c in v["competitors"]) + " | share basis: " + " / ".join(bases)
    if isinstance(v, dict) and isinstance(v.get("segments"), list) and v["segments"]:
        rest = {k: x for k, x in v.items() if k != "segments"}
        return "; ".join(f"{s['segment']} {s['share_pct']}%" for s in v["segments"]) + (" | " + fmt_value(rest, 1) if fmt_value(rest, 1) else "")
    if v is None: return "none"
    if isinstance(v, (int, float, str)): return str(v)
    if isinstance(v, list):
        return "; ".join(fmt_value(x, depth + 1) for x in v) if all(not isinstance(x, (dict, list)) for x in v) else " | ".join(fmt_value(x, depth + 1) for x in v)
    if isinstance(v, dict):
        parts = []
        for k, x in v.items():
            if x is None or x == [] or k in ("why_chosen", "scope_note", "internal_conflict", "concentration_note"): continue
            parts.append(f"{k.replace('_', ' ')}: {fmt_value(x, depth + 1)}")
        return ("; " if depth == 0 else ", ").join(parts)
    return str(v)

def best_citation(x):
    """The citation that carries the most of the signal's numbers (a human can check a number fastest); else the first."""
    want = {round(n, 6) for _, n in numeric_leaves(x["value"])}
    score = lambda c: len(want & {round(n, 6) for n in numbers_in(c["quote"])})
    return max(x["citations"], key=score)

def spot_check(records, seed=SPOT_SEED, k=5):
    cands = [(sid, x) for sid, r in records.items() for x in ([r["primary"]] if r["primary"] else []) + r["other_documents"] if x["citations"]]
    picks = random.Random(seed).sample(cands, min(k, len(cands)))
    out = []
    for sid, x in picks:
        c = best_citation(x)
        out.append({"signal_id": sid, "name": SIGNAL_META[sid][0], "doc_id": x["doc_id"], "pdf_page": c["pdf_page"], "printed_page": c["printed_page"], "quote": c["quote"]})
    return out

def build_brief(company, description, docs, naics_choice, extractions, records, gaps, summ, extractor_note, scope_table, verify_mode, family):
    signals = []
    for sid, r in records.items():
        p = r["primary"]
        signals.append({"signal_id": sid, "name": r["name"], "status": p["status"] if p else "NOT_FOUND",
                        "force_tag": p["force_tag"] if p else SIGNAL_META[sid][1], "primary": p, "other_documents": r["other_documents"],
                        "not_supported_by": r["not_supported_by"], "conflicts": r["conflicts"],
                        "quotes_verified": sum(len(x["citations"]) for x in ([p] if p else []) + r["other_documents"])})
    for sg in signals:
        for x in ([sg["primary"]] if sg["primary"] else []) + sg["other_documents"]:
            x["value_text"] = fmt_value(x["value"])
    return {
        "meta": {"company": company, "description": description, "generated": datetime.datetime.now().isoformat(timespec="seconds"),
                 "extractor": extractor_note, "extraction_done_by": "Claude Code (an LLM agent) in the build session; verification, reconciliation, gaps and this brief are code.",
                 "gemini_backend_status": GEMINI_TEST_STATUS, "verification_mode": verify_mode, "primary_rule": PRIMARY_RULE,
                 "proxy_industry_warning": f"Fractional CFO has no industry code. Every figure describes a proxy industry (chosen NAICS {naics_choice['chosen_code']}) or the accounting neighbor; see gaps.",
                 "hand_checked_line": "Hand-checked by me: [ ] of [ ] citations. (left blank on purpose; to be filled in by the student)"},
        "naics": naics_choice, "report_scope": scope_table,
        "sources": [{**{k: docs[d].get(k) for k in ("doc_id", "title", "publisher", "publication_date", "stated_industry_code", "how_to_find", "n_pages", "code_note")},
                     "industry_family": family[d], "naics_2022_listed": extractions[d]["report_scope"]["naics_2022_listed"]} for d in docs],
        "verification_summary": summ, "signals": signals, "gaps": gaps,
        "spot_check": {"seed": SPOT_SEED, "how": "random.Random(seed).sample over (signal, document) findings that have verified citations", "items": spot_check(records)},
    }

def cite_str(c):
    printed = f" (printed {c['printed_page']})" if c.get("printed_page") else ""
    return f"[{c['doc_id']}, {c['section_heading'] or 'no heading detected'}, PDF p.{c['pdf_page']}{printed}]"

def render_brief_md(b):
    L = [f"# Industry brief: {b['meta']['company']}", "", f"*{b['meta']['description']}*", "",
         f"**Extraction:** {b['meta']['extraction_done_by']}", f"**Verification mode this run:** {b['meta']['verification_mode']}", "",
         f"> {b['meta']['proxy_industry_warning']}", "", "## NAICS selected", "",
         f"**{b['naics']['chosen_code']} {b['naics']['chosen_title']}** (2022 NAICS Manual: {b['naics']['manual_url']}, accessed {b['naics']['accessed']}).", "",
         b["naics"]["justification"], "", "## Verification summary", ""]
    s = b["verification_summary"]
    L += [f"Quotes checked {s['quotes_checked']}; passed {s['passed']} (exact {s['passed_exact']}, adjacent page {s['passed_adjacent_page']}, fuzzy {s['passed_fuzzy']}); failed {s['failed']}; "
          f"corrected in retry {s['corrected_in_retry']}; numbers checked {s['numbers_checked']}, unmatched {s['numbers_unmatched']}.", "", "## Signals", ""]
    for sg in b["signals"]:
        L += [f"### {sg['signal_id']} {sg['name']}", f"*force: {sg['force_tag']} | status: {sg['status']} (for the proxy industry, not the fractional niche) | quotes verified: {sg['quotes_verified']}*", ""]
        for i, x in enumerate(([sg["primary"]] if sg["primary"] else []) + sg["other_documents"]):
            L += [f"**{'Primary' if i == 0 else 'Also'}: {x['doc_id']} ({x['status']}, confidence {x['confidence']['level']})**", f"- Value: {x['value_text']}", f"- Summary: {x['summary']}"]
            L += [f"- Cite {cite_str(c)} \"{c['quote']}\" ({c['verification']})" for c in x["citations"]]
            L += [f"- Note: {x['notes']}", ""]
        for c in sg["conflicts"]:
            L += [f"**Conflict ({c['kind']}): {c['metric']}**"] + [f"- {t['doc_id']}: {t['value']} ({t['period_or_year']})" + (f" {cite_str(t['citation'])} \"{t['citation']['quote']}\"" if t["citation"] else "") for t in c["sides"]] + [f"- Why they may differ: {c['why_they_may_differ']}", ""]
    L += ["## Sources", ""] + [f"- **{x['title']}**, {x['publisher']}, {x['publication_date']}, code: {x['stated_industry_code'] or 'not stated'}. Find it: {x['how_to_find']}" for x in b["sources"]]
    L += ["", "## Gaps", ""] + [f"- **{g['why']}**" + (f" ({g['signal_id']}, {g['doc_id']})" if g["doc_id"] else "") + f": {g['what_is_missing']} *To find it:* {g['what_would_find_it']}" for g in b["gaps"]]
    L += ["", "## Spot-check list (5 random findings, seed %d)" % b["spot_check"]["seed"], ""] + \
         [f"- {x['signal_id']} {x['name']}: {x['doc_id']}, PDF p.{x['pdf_page']} (printed {x['printed_page']}): \"{x['quote']}\"" for x in b["spot_check"]["items"]]
    L += ["", b["meta"]["hand_checked_line"], ""]
    return "\n".join(L)

BRIEF = build_brief(COMPANY, DESCRIPTION, DOCS, NAICS_CHOICE, EXTRACTIONS, RECORDS, GAPS, SUMM, EXTRACTOR_NOTE, SCOPE_TABLE, VERIFY_MODE, FAMILY)
json.dump(BRIEF, open("brief.json", "w"), indent=1, ensure_ascii=False)
open("brief.md", "w").write(render_brief_md(BRIEF))
print("Wrote brief.json and brief.md")
print("Signals:", {s["signal_id"]: s["status"] for s in BRIEF["signals"]})
print("\nSpot-check list (compare these 5 against the PDFs):")
for x in BRIEF["spot_check"]["items"]:
    print(f"  {x['signal_id']} | {x['doc_id']} | PDF p.{x['pdf_page']} (printed {x['printed_page']}) | \"{x['quote'][:80]}\"")
print("\n--- brief.md, first lines ---")
print("\n".join(open("brief.md").read().split("\n")[:22]))

Wrote brief.json and brief.md
Signals: {'S1': 'FOUND', 'S2': 'FOUND', 'S3': 'FOUND', 'S4': 'FOUND', 'S5': 'FOUND', 'S6': 'FOUND', 'S7': 'FOUND'}

Spot-check list (compare these 5 against the PDFs):
  S7 | firstresearch_accounting-services | PDF p.1 (printed 1) | "Risk: Slow economy cuts business needs"
  S2 | ibisworld_54161 | PDF p.34 (printed 31) | "Accenture Plc 2.5–5 10,000+ EY 2.5–5 10,000+ PwC 2.5–5 10,000+ KPMG 0–2.5 10,000"
  S4 | firstresearch_accounting-services | PDF p.14 (printed 14) | "Accounting firms depend heavily on the reputation and expertise of senior partne"
  S2 | ibisworld_54121c | PDF p.32 (printed 29) | "Pwc ($15.4bn) 9.7% Deloitte ($15.1bn) 9.5%"
  S4 | ibisworld_54121c | PDF p.42 (printed 39) | "median accountant salary, which rose from $81,680 in 2024 to $83,680 in 2025"

--- brief.md, first lines ---
# Industry brief: Working name TBD, fractional CFO services

*Fractional (part-time, outsourced) CFO services for US small and mid-sized businesses with roughl

## `run_pipeline(company, description, reports_dir, extractor)`: the whole thing in one call

**Choice:** one function chains the same stage functions used above, so a reader can run it on a new industry by editing the config cell, adding PDFs to `reports/`, and either supplying
`extractions/*.json` (the default, `claude_code_files`) or setting `extractor="gemini"` with a free key. The cell below calls it and then checks that it reproduces the brief the step-by-step cells built.

**Alternative rejected:** hiding the stages inside the function. The stages are visible above so each choice can be inspected; the function only sequences them.
**Limit:** the NAICS choice and the extraction files are industry-specific. Changing the industry means re-doing those two human/agent steps; the other stages will re-run unchanged.

In [17]:
def run_pipeline(company, description, reports_dir, extractor="claude_code_files", write_files=True):
    """Stage 0 ingest (if PDFs present) -> 1 NAICS fetch+validate -> 2 extract -> 3 verify -> 4 reconcile -> 5 gaps -> brief. Returns the brief dict."""
    if os.environ.get("ANTHROPIC_API_KEY"):
        raise RuntimeError("ANTHROPIC_API_KEY is set; this pipeline is zero-cost and never calls a paid API. Unset it.")
    if glob.glob(os.path.join(reports_dir, "*.pdf")):
        pages, docs, _ = stage0_ingest(reports_dir)
    else:
        pages, docs = [], (json.load(open("sources.json")) if os.path.exists("sources.json") else {})
    naics = stage1_fetch()
    choice = stage1_choose(naics)
    extractions, note = run_extractor(extractor, pages, docs)
    items, numrows, struct, log, mode = stage3_verify(extractions)
    summ = summarize(items, numrows, struct)
    r1 = {tuple(k) for k in (log.get("rounds") or [{"failed_keys": []}])[0]["failed_keys"]}
    summ["corrected_in_retry"] = len(r1 & {(i["doc_id"], i["signal_id"], i["evidence_index"]) for i in items if i["result"] != "FAILED"})
    summ["rounds_run"] = len(log.get("rounds", []))
    verified = apply_verification(extractions, items, numrows)
    records, family = stage4_reconcile(extractions, verified, items, docs)
    gaps = stage5_gaps(extractions, verified, records, docs, naics=naics, chosen=choice["chosen_code"])
    brief = build_brief(company, description, docs, choice, extractions, records, gaps, summ, note, report_scope_table(extractions, docs, choice["chosen_code"]), mode, family)
    if write_files:
        json.dump(brief, open("brief.json", "w"), indent=1, ensure_ascii=False)
        open("brief.md", "w").write(render_brief_md(brief))
    return brief

BRIEF2 = run_pipeline(COMPANY, DESCRIPTION, REPORTS_DIR, EXTRACTOR)
_strip = lambda b: json.dumps({k: v for k, v in b.items() if k != "meta"}, sort_keys=True)
assert _strip(BRIEF2) == _strip(BRIEF), "run_pipeline() must reproduce the step-by-step brief"
print("run_pipeline() reproduced the step-by-step brief exactly (everything except the timestamp).\n")
print("FINAL SUMMARY")
print("  company        :", BRIEF2["meta"]["company"])
print("  NAICS          :", BRIEF2["naics"]["chosen_code"], BRIEF2["naics"]["chosen_title"])
print("  extractor      :", BRIEF2["meta"]["extractor"])
print("  gemini backend :", BRIEF2["meta"]["gemini_backend_status"])
print("  verification   :", BRIEF2["meta"]["verification_mode"])
_s = BRIEF2["verification_summary"]
print(f"  quotes         : {_s['quotes_checked']} checked, {_s['passed']} passed, {_s['failed']} failed, {_s['corrected_in_retry']} corrected in retry; numbers {_s['numbers_checked']} checked, {_s['numbers_unmatched']} unmatched")
print("  signal status  :", {s['signal_id']: s['status'] for s in BRIEF2['signals']})
print("  conflicts      :", sum(len(s['conflicts']) for s in BRIEF2['signals']), "| gaps:", len(BRIEF2['gaps']))

run_pipeline() reproduced the step-by-step brief exactly (everything except the timestamp).

FINAL SUMMARY
  company        : Working name TBD, fractional CFO services
  NAICS          : 541611 Administrative Management and General Management Consulting Services
  extractor      : claude_code_files: Claude Code (LLM agent) wrote these files in the build session
  gemini backend : implemented, untested (never called the live Gemini API; logic exercised only with a mock)
  verification   : LIVE: every quote re-checked against the PDF page text just extracted
  quotes         : 79 checked, 79 passed, 0 failed, 0 corrected in retry; numbers 55 checked, 0 unmatched
  signal status  : {'S1': 'FOUND', 'S2': 'FOUND', 'S3': 'FOUND', 'S4': 'FOUND', 'S5': 'FOUND', 'S6': 'FOUND', 'S7': 'FOUND'}
  conflicts      : 11 | gaps: 15
